In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import sys

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

from src.analytics.ratios import compute_all_ratios
from src.analytics.cagr import compute_all_cagr
from src.analytics.cashflow_kpis import compute_all_cashflow_kpis

print("All analytics modules imported!")

All analytics modules imported!


In [2]:
print("Step 1/3 — Computing profitability/leverage/efficiency ratios...")
ratios_df = compute_all_ratios()

print("\nStep 2/3 — Computing CAGR growth ratios...")
cagr_df = compute_all_cagr()

print("\nStep 3/3 — Computing cash flow KPIs...")
cashflow_df = compute_all_cashflow_kpis()

print("\n\nAll 3 KPI sets computed!")
print(f"  Ratios shape:    {ratios_df.shape}")
print(f"  CAGR shape:      {cagr_df.shape}")
print(f"  Cash Flow shape: {cashflow_df.shape}")

Step 1/3 — Computing profitability/leverage/efficiency ratios...
Loading merged data...
Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Computing profitability ratios...
Computing leverage ratios...
Computing efficiency ratios...

Ratio Engine Complete!
  Total rows:    1082
  Companies:     93
  Year range:    2011-03 to 2024-09
  Ratios computed: 12

Step 2/3 — Computing CAGR growth ratios...
Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols


In [3]:
# Ratios and Cash Flow KPIs are both at company_id + year level
# CAGR is at company_id level only (one row per company)

# First merge ratios with cashflow on company_id + year
master = pd.merge(
    ratios_df,
    cashflow_df,
    on=['company_id', 'year'],
    how='outer'
)

# Then merge CAGR (company level) onto every year for that company
master = pd.merge(
    master,
    cagr_df,
    on='company_id',
    how='left'
)

print(f"Master financial_ratios table shape: {master.shape}")
print(f"Total columns: {len(master.columns)}")
print(f"\nColumn list:")
for col in master.columns:
    print(f"  - {col}")

Master financial_ratios table shape: (1159, 43)
Total columns: 43

Column list:
  - company_id
  - year
  - net_profit_margin_pct
  - operating_profit_margin_pct
  - return_on_equity_pct
  - return_on_capital_pct
  - debt_to_equity
  - interest_coverage
  - net_debt
  - asset_turnover
  - fixed_asset_turnover
  - ebit
  - capital_employed
  - total_equity
  - operating_activity
  - investing_activity
  - financing_activity
  - free_cash_flow_cr
  - cfo_quality_score
  - cfo_quality_label
  - capex_cr
  - capex_intensity_pct
  - capex_label
  - cf_pattern
  - cf_label
  - sales_cagr_3yr
  - sales_cagr_3yr_flag
  - sales_cagr_5yr
  - sales_cagr_5yr_flag
  - sales_cagr_10yr
  - sales_cagr_10yr_flag
  - net_profit_cagr_3yr
  - net_profit_cagr_3yr_flag
  - net_profit_cagr_5yr
  - net_profit_cagr_5yr_flag
  - net_profit_cagr_10yr
  - net_profit_cagr_10yr_flag
  - eps_cagr_3yr
  - eps_cagr_3yr_flag
  - eps_cagr_5yr
  - eps_cagr_5yr_flag
  - eps_cagr_10yr
  - eps_cagr_10yr_flag


In [4]:
# Sort for readability
master = master.sort_values(['company_id', 'year']).reset_index(drop=True)

# Check for duplicate columns (in case ratios.py and cashflow_kpis.py
# both produced overlapping fields)
print("Checking for any unexpected duplicate rows...")
dupes = master[master.duplicated(subset=['company_id', 'year'])]
print(f"  Duplicate (company_id, year) pairs: {len(dupes)}")

print(f"\nFinal master table:")
print(f"  Rows: {len(master)}")
print(f"  Companies: {master['company_id'].nunique()}")
print(f"  Year range: {master['year'].min()} to {master['year'].max()}")

Checking for any unexpected duplicate rows...
  Duplicate (company_id, year) pairs: 0

Final master table:
  Rows: 1159
  Companies: 100
  Year range: 2011-03 to 2024-09


In [5]:
# Connect to database
conn = sqlite3.connect('data/nifty100.db')

# Save as financial_ratios table (replacing the original supplementary one
# with our fully computed version)
master.to_sql('financial_ratios_computed', conn, if_exists='replace', index=False)

print("Saved to database as 'financial_ratios_computed' table!")

# Verify
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM financial_ratios_computed")
row_count = cursor.fetchone()[0]
print(f"Row count in database: {row_count}")

conn.close()
print("\nDatabase connection closed.")

Saved to database as 'financial_ratios_computed' table!
Row count in database: 1159

Database connection closed.


In [6]:
# Reconnect and run a sample query to prove it works end to end
conn = sqlite3.connect('data/nifty100.db')

query = """
    SELECT company_id, year, 
           return_on_equity_pct, 
           debt_to_equity, 
           free_cash_flow_cr,
           sales_cagr_5yr,
           cf_label
    FROM financial_ratios_computed
    WHERE company_id = 'TCS'
    ORDER BY year DESC
    LIMIT 5
"""

result = pd.read_sql_query(query, conn)
print("TCS — Last 5 years from financial_ratios_computed table:")
print(result.to_string(index=False))

conn.close()

TCS — Last 5 years from financial_ratios_computed table:
company_id    year  return_on_equity_pct  debt_to_equity  free_cash_flow_cr  sales_cagr_5yr     cf_label
       TCS 2024-03                 50.94            0.09            50429.0           10.46 Asset Seller
       TCS 2023-03                 46.78            0.09            42513.0           10.46 Asset Seller
       TCS 2022-03                 43.13            0.09            39211.0           10.46   Reinvestor
       TCS 2021-03                 37.67            0.09            30846.0           10.46   Reinvestor
       TCS 2020-03                 38.57            0.10            41337.0           10.46 Asset Seller
